# Hafta 3 - Veri çekme: Bir MWh'nin yolculuğu

**Soru:** Türkiye'de elektriğin ne kadarı PTF'den fiyatlanıyor, bu pay mevsime ve yıllara göre nasıl değişiyor ve değişimi ne sürüklüyor?

| Veri seti (EPİAŞ Şeffaflık) | eptr2 anahtarı | Ne için |
|---|---|---|
| İkili anlaşma alış / satış miktarı (saatlik) | `bi-long`, `bi-short` | İkili anlaşma hacmi |
| EÜAŞ - GTŞ ikili anlaşmaları (aylık) | `bi-euas` | Düzenlemeye tabi kısım |
| GÖP eşleşme miktarı (saatlik) | `dam-clearing` | GÖP hacmi |
| GİP eşleşme miktarı | `idm-qty` | GİP hacmi |
| YAL / YAT talimat miktarları | `bpm-up`, `bpm-down` | Dengeleme hacmi |
| YEKDEM UEVM (kaynak bazında, saatlik) | `ren-uevm` | GÖP'e mekanik giren YEKDEM üretimi |
| Elektrik piyasa hacimleri (fiziksel) | `electricity-market-quantity` | EPİAŞ'ın kendi özetiyle çapraz kontrol |

PTF, SMF, GİP fiyatı, tüketim ve üretim Hafta 2'den (`../Hafta_02/02_islenmis/ana_tablo_h02_saatlik.csv`) alınır, yeniden çekilmez.

**Çalıştırma:** Bu dosya `Hafta_03` klasöründe. *Kernel → Restart & Run All*. Önce her veri seti için 2 günlük deneme yapılır ve sütunlar yazdırılır, sonra 2023-2026 aylık parçalar halinde çekilir. Çekilen aylar `01_ham_veri/` altına kaydedilir; yarıda kalırsa yeniden çalıştırınca kaldığı yerden devam eder. Süre: yaklaşık 15-25 dakika.

API çağrıları için açık kaynak [`eptr2`](https://github.com/tideseed/eptr2) paketi kullanılır (uç nokta yolları ve yanıt ayrıştırma paket içinde).

In [ ]:
# 1) Kurulum ve ayarlar
import sys, subprocess
try:
    import eptr2
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "eptr2"])
    import eptr2
import os, time, getpass, warnings
from datetime import date, timedelta
from pathlib import Path
import numpy as np, pandas as pd
from eptr2 import EPTR2
warnings.filterwarnings("ignore")

BAS = date(2023, 1, 1)
BIT = date.today() - timedelta(days=1)
HAM  = Path("01_ham_veri"); HAM.mkdir(exist_ok=True)
ISLN = Path("02_islenmis"); ISLN.mkdir(exist_ok=True)
H02_TABLO = Path("../Hafta_02/02_islenmis/ana_tablo_h02_saatlik.csv")
print("eptr2 sürümü:", getattr(eptr2, "__version__", "?"), "| Aralık:", BAS, "-", BIT)
print("Hafta 2 tablosu bulundu mu:", H02_TABLO.exists())

In [ ]:
# 2) Giriş (kullanıcı adı = EPİAŞ şeffaflık kayıt e-postası; bilgiler kaydedilmez)
kullanici = os.environ.get("EPIAS_USER") or input("EPİAŞ kullanıcı adı (e-posta): ").strip()
sifre     = os.environ.get("EPIAS_PASS") or getpass.getpass("EPİAŞ şifre: ")
eptr = EPTR2(username=kullanici, password=sifre, use_dotenv=False, recycle_tgt=True)
del sifre
test = eptr.call("mcp", start_date="2026-10-01", end_date="2026-10-01")
print("Giriş başarılı. Deneme PTF satırı:", len(test))

In [ ]:
# 3) Çağrı yardımcıları
ANAHTARLAR = {
    "ia_alis":  "bi-long",
    "ia_satis": "bi-short",
    "gop":      "dam-clearing",
    "gip":      "idm-qty",
    "yal":      "bpm-up",
    "yat":      "bpm-down",
    "yekdem":   "ren-uevm",
    "euas_gts": "bi-euas",
    "hacim":    "electricity-market-quantity",
}

def cek(anahtar, b, s, deneme=3):
    for i in range(deneme):
        try:
            df = eptr.call(anahtar, start_date=str(b), end_date=str(s))
            return df if isinstance(df, pd.DataFrame) else pd.DataFrame(df)
        except Exception as e:
            hata = str(e)[:300]
            if i == deneme - 1:
                print(f"    HATA {anahtar} {b}..{s}: {hata}")
                return None
            time.sleep(3 * (i + 1))

def aylar():
    d = date(BAS.year, BAS.month, 1)
    while d <= BIT:
        son = date(d.year + d.month // 12, d.month % 12 + 1, 1) - timedelta(days=1)
        yield d, min(son, BIT), son > BIT
        d = son + timedelta(days=1)

## 4) Deneme: her veri seti için 2 gün
Bu hücrenin çıktısı sütun adlarını gösterir. Bir veri seti hata verirse not alınır, çekim diğerleriyle devam eder.

In [ ]:
DENEME = {}
with open(HAM / "deneme_kolonlar.txt", "w", encoding="utf-8") as f:
    for ad, k in ANAHTARLAR.items():
        b, s = (date(2026, 9, 1), date(2026, 9, 30)) if ad in ("euas_gts", "hacim") else (date(2026, 10, 1), date(2026, 10, 2))
        df = cek(k, b, s)
        DENEME[ad] = df
        satir = f"{ad:9s} ({k}): " + ("HATA / boş" if df is None or df.empty else f"{df.shape} -> {list(df.columns)}")
        print(satir); f.write(satir + "\n")
        if df is not None and not df.empty:
            f.write(df.head(3).to_string() + "\n\n")
calisan = [ad for ad, df in DENEME.items() if df is not None and not df.empty]
print("\nÇalışan veri setleri:", calisan)

## 5) Tam çekim (2023-01 → dün), aylık parçalar
Tamamlanmış aylar `01_ham_veri/<ad>_<YYYY-MM>.csv` olarak saklanır ve tekrar çekilmez.

In [ ]:
HAM_VERI = {}
for ad in calisan:
    k = ANAHTARLAR[ad]; parcalar = []
    print(f"\n[{ad}] {k}")
    for b, s, kismi in aylar():
        dosya = HAM / f"{ad}_{b:%Y-%m}{'_kismi' if kismi else ''}.csv"
        if dosya.exists() and not kismi:
            df = pd.read_csv(dosya)
        else:
            df = cek(k, b, s)
            if df is not None and not df.empty:
                df.to_csv(dosya, index=False, encoding="utf-8-sig")
            time.sleep(0.5)
        n = 0 if df is None else len(df)
        if n: parcalar.append(df)
        print(f"  {b:%Y-%m}: {n}", end=" |")
    HAM_VERI[ad] = pd.concat(parcalar, ignore_index=True) if parcalar else pd.DataFrame()
    print(f"\n  toplam: {len(HAM_VERI[ad])} satır")

## 6) Saatlik tabloya dönüştür
Tarih sütunu bulunur, sayısal sütunlar saat bazında toplanır. GİP'te blok kontratlar ayrılır (yalnızca saatlik kontratlar; blok hacmi küçüktür ve ayrıca not edilir).

In [ ]:
def zaman_sutunu(df):
    for c in ["date", "tarih", "deliveryDate", "period", "donem"]:
        if c in df.columns:
            return pd.to_datetime(df[c], utc=True, errors="coerce").dt.tz_convert("Europe/Istanbul").dt.tz_localize(None)
    if "kontratAdi" in df.columns:   # PHyyMMddhh
        k = df["kontratAdi"].astype(str)
        return pd.to_datetime(k.str[2:10], format="%y%m%d%H", errors="coerce")
    raise KeyError(f"Tarih sütunu bulunamadı: {list(df.columns)}")

ATLA = {"hour", "id", "saat", "hourly"}
def saatlik(df, on_ek):
    df = df.copy()
    blok = None
    if "kontratAdi" in df.columns:
        blok_mask = ~df["kontratAdi"].astype(str).str.startswith("PH")
        blok = df[blok_mask]; df = df[~blok_mask]
    df["zaman"] = zaman_sutunu(df).dt.floor("h")
    sayisal = [c for c in df.columns if c != "zaman" and c.lower() not in ATLA and pd.api.types.is_numeric_dtype(df[c])]
    out = df.groupby("zaman")[sayisal].sum(min_count=1)
    out.columns = [f"{on_ek}_{c}" for c in out.columns]
    return out, blok

SAATLIK, AYLIK_HAM = [], {}
for ad, df in HAM_VERI.items():
    if df.empty: continue
    if ad in ("euas_gts", "hacim"):          # aylık/günlük özet setler olduğu gibi saklanır
        AYLIK_HAM[ad] = df; df.to_csv(ISLN / f"h03_{ad}.csv", index=False, encoding="utf-8-sig")
        print(f"{ad}: {df.shape} kaydedildi"); continue
    s, blok = saatlik(df, ad)
    SAATLIK.append(s)
    print(f"{ad}: {s.shape[0]} saat, sütunlar {list(s.columns)}" + (f" | ayrılan blok satır: {len(blok)}" if blok is not None and len(blok) else ""))

h03 = pd.concat(SAATLIK, axis=1).sort_index()
h03 = h03[~h03.index.duplicated()]
print("\nYeni saatlik tablo:", h03.shape)

## 7) Hafta 2 tablosuyla birleştir (PTF, SMF, GİP fiyatı, tüketim, üretim)

In [ ]:
ana = pd.read_csv(H02_TABLO, parse_dates=["zaman"]).set_index("zaman")
ana = ana[~ana.index.duplicated()]
tablo = ana.join(h03, how="left")
tablo.to_csv(ISLN / "ana_tablo_h03_saatlik.csv", encoding="utf-8-sig")
print("Birleşik tablo:", tablo.shape, "| aralık:", tablo.index.min(), "-", tablo.index.max())
eksik = tablo[h03.columns].isna().mean().sort_values(ascending=False)
print("\nYeni sütunlarda eksik oran:"); print((eksik * 100).round(1).to_string())

## 8) Hızlı kontrol: aylık hacimlerin tüketime oranı
Bu yalnızca ilk bakış; analiz ayrıca yapılacak. Rakamlar mantıksız görünürse (ör. %0 ya da %500) sütun eşleşmesi kontrol edilecek.

In [ ]:
tuk = [c for c in tablo.columns if c.lower().startswith(("tuketim", "consumption"))]
print("Tüketim sütunu:", tuk[:3])
ay = tablo.resample("MS").sum(numeric_only=True)
oz = pd.DataFrame(index=ay.index)
if tuk:
    T = ay[tuk[0]]
    for c in h03.columns:
        oz[c] = (ay[c] / T * 100).round(1)
oz.to_csv(ISLN / "h03_aylik_tuketime_oran.csv", encoding="utf-8-sig")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
display(oz.tail(14))
print("\nBitti. Bana 'bitti' yazmanız yeterli; 02_islenmis/ klasöründeki dosyaları okuyup analize geçeceğim.")